# Day 16: Qdrant CRUD Operations

## Core Theory (Just-in-Time)
In AI Engineering, a Vector Database like Qdrant acts as the foundational memory layer for Retrieval-Augmented Generation (RAG). While creating collections and querying vectors are common tasks, understanding how to manage the lifecycle of those vectors (Create, Read, Update, Delete - CRUD) with precise metadata is critical for building robust production systems.

### The "Why"
- **Dynamic Data:** Real-world knowledge bases are not static. Documents are updated, user preferences change, and outdated information must be removed to prevent hallucinations.
- **Metadata Filtering:** Vectors alone just provide semantic similarity. Combining them with structured metadata (payloads in Qdrant) enables complex hybrid queries (e.g., 'find similar documents WHERE author=X AND status=active').
- **Consistency:** Ensuring that updates to source documents correctly reflect in the vector store is a major challenge in production AI.

### The "How"
We use the `qdrant-client` library in Python to interact with a Qdrant instance. Operations like `upsert` (Create/Update) and `delete` target specific points (vectors) using unique IDs. Payloads are JSON-like objects attached to these points. We will use `pydantic` to enforce strict schemas for our data before it hits the vector database.


## Code Implementation

The following code demonstrates how to define a schema, initialize an in-memory Qdrant client, and perform Create, Read, Update, and Delete operations using strict type hinting and docstrings.


In [1]:
import uuid
from typing import List, Optional, Any, Dict
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance, UpdateResult

class DocumentMetadata(BaseModel):
    """Strict schema for the metadata attached to a vector."""
    title: str = Field(..., description="The title of the document")
    author: str = Field(..., description="The author of the document")
    status: str = Field(default="active", description="Document status (active/archived)")
    
class QdrantManager:
    """Manages CRUD operations for a specific Qdrant collection."""
    
    def __init__(self, collection_name: str, vector_size: int = 4):
        """
        Initialize the Qdrant client and ensure the collection exists.
        For this demonstration, we use an in-memory instance.
        """
        self.client = QdrantClient(":memory:")
        self.collection_name = collection_name
        self.vector_size = vector_size
        
        # Create collection if it doesn't exist (simplified for in-memory)
        if not self.client.collection_exists(collection_name=self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=VectorParams(size=self.vector_size, distance=Distance.COSINE)
            )
            print(f"Collection '{self.collection_name}' created.")

    def insert_point(self, point_id: str, vector: List[float], payload: DocumentMetadata) -> None:
        """
        Creates (or overwrites) a vector point with its associated metadata payload.
        """
        point = PointStruct(
            id=point_id,
            vector=vector,
            payload=payload.model_dump()
        )
        operation_info = self.client.upsert(
            collection_name=self.collection_name,
            points=[point]
        )
        print(f"Inserted/Upserted point {point_id}. Status: {operation_info.status.name}")

    def read_point(self, point_id: str) -> Optional[Dict[str, Any]]:
        """
        Retrieves a point's payload and vector by its ID.
        """
        points = self.client.retrieve(
            collection_name=self.collection_name,
            ids=[point_id],
            with_payload=True,
            with_vectors=True
        )
        if not points:
            print(f"Point {point_id} not found.")
            return None
            
        print(f"Retrieved point {point_id}: Payload = {points[0].payload}")
        return points[0].payload

    def update_payload(self, point_id: str, new_payload: dict) -> None:
        """
        Updates specific fields in the metadata payload without modifying the vector.
        """
        operation_info = self.client.set_payload(
            collection_name=self.collection_name,
            payload=new_payload,
            points=[point_id]
        )
        print(f"Updated payload for point {point_id}. Status: {operation_info.status.name}")

    def delete_point(self, point_id: str) -> None:
        """
        Completely removes a point (vector + payload) from the collection.
        """
        operation_info = self.client.delete(
            collection_name=self.collection_name,
            points_selector=[point_id]
        )
        print(f"Deleted point {point_id}. Status: {operation_info.status.name}")

# Example Execution
if __name__ == "__main__":
    manager = QdrantManager(collection_name="engineering_docs")
    
    # 1. Create (Insert)
    doc_id = str(uuid.uuid4())
    metadata = DocumentMetadata(title="Qdrant Best Practices", author="Alice")
    dummy_vector = [0.1, 0.2, 0.3, 0.4] # Mock embedding
    
    print("\n--- Create ---")
    manager.insert_point(point_id=doc_id, vector=dummy_vector, payload=metadata)
    
    # 2. Read
    print("\n--- Read ---")
    manager.read_point(point_id=doc_id)
    
    # 3. Update
    print("\n--- Update ---")
    manager.update_payload(point_id=doc_id, new_payload={"status": "archived", "tags": ["db", "vector"]})
    manager.read_point(point_id=doc_id) # Verify update
    
    # 4. Delete
    print("\n--- Delete ---")
    manager.delete_point(point_id=doc_id)
    manager.read_point(point_id=doc_id) # Verify deletion


Collection 'engineering_docs' created.

--- Create ---
Inserted/Upserted point 70d18bc8-ce59-4f0f-a955-1174977e99f8. Status: COMPLETED

--- Read ---
Retrieved point 70d18bc8-ce59-4f0f-a955-1174977e99f8: Payload = {'title': 'Qdrant Best Practices', 'author': 'Alice', 'status': 'active'}

--- Update ---
Updated payload for point 70d18bc8-ce59-4f0f-a955-1174977e99f8. Status: COMPLETED
Retrieved point 70d18bc8-ce59-4f0f-a955-1174977e99f8: Payload = {'title': 'Qdrant Best Practices', 'author': 'Alice', 'status': 'archived', 'tags': ['db', 'vector']}

--- Delete ---
Deleted point 70d18bc8-ce59-4f0f-a955-1174977e99f8. Status: COMPLETED
Point 70d18bc8-ce59-4f0f-a955-1174977e99f8 not found.


## Common Pitfalls in Production

1. **Dangling Vectors (Zombie Data):** Deleting a document from your primary database (e.g., Postgres) but forgetting to issue a delete command to Qdrant. This leads to the LLM answering questions based on deleted or revoked information.
2. **Schema Drift:** Modifying the structure of the metadata payloads in your application without ensuring historical payloads in the vector DB are compatible, causing crashes during filtering or retrieval.
3. **Costly Complete Updates:** Re-embedding the entire document text just to update a small metadata field (like `status=active` to `status=archived`). You should use Qdrant's payload update mechanisms (`set_payload`) instead of full `upsert` when the text hasn't changed.


## Practical Lab / Homework

**Task:** Expand upon the CRUD operations by implementing a batch upsert mechanism.
1. Create a method `batch_insert_points` in a new class `BatchQdrantManager`.
2. It should accept a list of IDs, a list of vectors, and a list of `DocumentMetadata` objects.
3. Execute a single batch upsert call to Qdrant.
4. Verify the operation by retrieving all inserted points.

**Constraint:** Provide a fully working script with strict type hinting. Do not use pseudo-code.


In [2]:
from qdrant_client.models import Batch

class BatchQdrantManager(QdrantManager):
    """Extends the QdrantManager to support batch operations."""
    
    def batch_insert_points(self, point_ids: List[str], vectors: List[List[float]], payloads: List[DocumentMetadata]) -> None:
        """
        Performs a batch upsert of multiple points simultaneously.
        """
        if not (len(point_ids) == len(vectors) == len(payloads)):
            raise ValueError("Mismatched lengths of IDs, vectors, and payloads.")
            
        # Convert Pydantic models to dicts
        payload_dicts = [p.model_dump() for p in payloads]
        
        operation_info = self.client.upsert(
            collection_name=self.collection_name,
            points=Batch(
                ids=point_ids,
                vectors=vectors,
                payloads=payload_dicts
            )
        )
        print(f"Batch upserted {len(point_ids)} points. Status: {operation_info.status.name}")

    def read_multiple_points(self, point_ids: List[str]) -> List[Dict[str, Any]]:
        """
        Retrieves multiple points by their IDs.
        """
        points = self.client.retrieve(
            collection_name=self.collection_name,
            ids=point_ids,
            with_payload=True
        )
        
        results = []
        for p in points:
            print(f"Retrieved point {p.id}: Payload = {p.payload}")
            if p.payload:
                results.append(p.payload)
        return results

# Lab Solution Execution
if __name__ == "__main__":
    batch_manager = BatchQdrantManager(collection_name="batch_engineering_docs")
    
    # Generate dummy data
    ids = [str(uuid.uuid4()) for _ in range(3)]
    vecs = [
        [0.1, 0.1, 0.1, 0.1],
        [0.2, 0.2, 0.2, 0.2],
        [0.3, 0.3, 0.3, 0.3]
    ]
    docs = [
        DocumentMetadata(title="Doc A", author="Alice"),
        DocumentMetadata(title="Doc B", author="Bob"),
        DocumentMetadata(title="Doc C", author="Charlie")
    ]
    
    print("\n--- Batch Create ---")
    batch_manager.batch_insert_points(point_ids=ids, vectors=vecs, payloads=docs)
    
    print("\n--- Verify Batch Insertion ---")
    batch_manager.read_multiple_points(point_ids=ids)


Collection 'batch_engineering_docs' created.

--- Batch Create ---
Batch upserted 3 points. Status: COMPLETED

--- Verify Batch Insertion ---
Retrieved point 02fb122f-cccf-42f7-8381-9ff1104aa76d: Payload = {'title': 'Doc A', 'author': 'Alice', 'status': 'active'}
Retrieved point ce831205-d288-4b32-aa3c-8227245ea8a6: Payload = {'title': 'Doc B', 'author': 'Bob', 'status': 'active'}
Retrieved point 9aeccb75-f410-4bef-942d-4e520ea274df: Payload = {'title': 'Doc C', 'author': 'Charlie', 'status': 'active'}
